# Step 3 — Sim comparison (this week’s incomplete work)

Friendly notebook entrypoint to finish sim **before Jetson AGX Thor / G1 next week**.

1. **Offline** ZOH / linear / PID vs demo joints  
2. **Live UnifoLM** dual-process for each bridge: `esn`, `zoh`, `linear`  

Set `MOCK=False` for paper numbers. `MOCK=True` is only a smoke test.


In [ ]:
from pathlib import Path
import os
import sys

os.environ.setdefault("MUJOCO_GL", "egl")

NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name == "notebooks":
    RESEARCH_DIR = NOTEBOOK_DIR.parent
elif (NOTEBOOK_DIR / "src" / "step3_sim_comparison.py").is_file():
    RESEARCH_DIR = NOTEBOOK_DIR
else:
    RESEARCH_DIR = NOTEBOOK_DIR / "research_summer_2026" / "research"
    if not RESEARCH_DIR.is_dir():
        RESEARCH_DIR = NOTEBOOK_DIR / "research"

RESEARCH_DIR = RESEARCH_DIR.resolve()
assert (RESEARCH_DIR / "src").is_dir(), f"src package not found under: {RESEARCH_DIR}"

os.chdir(RESEARCH_DIR)
if str(RESEARCH_DIR) not in sys.path:
    sys.path.insert(0, str(RESEARCH_DIR))

print(f"Research root : {RESEARCH_DIR}")


In [ ]:
# ── Configuration ───────────────────────────────────────────
EPISODE = 0
DURATION_S = 10.0
MOCK = False                          # False = live UnifoLM (paper)
RUN_OFFLINE = True                    # ZOH/linear/PID table
RUN_LIVE = True                       # dual-process MuJoCo bridges
BRIDGES = ["esn", "zoh", "linear"]    # order to run live
RECORD_VIDEO = False
DEVICE = "cuda"

print(f"MOCK={MOCK} | offline={RUN_OFFLINE} | live={RUN_LIVE} | bridges={BRIDGES}")
if MOCK:
    print("WARNING: MOCK=True — results are not paper-ready.")


In [ ]:
# Cell A — Offline baselines (no UnifoLM download)
import json
from dataclasses import asdict

import pandas as pd
from IPython.display import display

from src.paths import results_path
from src.step3_control_baselines import run_all_baselines, write_comparison_table

offline_csv = None
if RUN_OFFLINE:
    offline_dir = results_path("step1_baselines")
    eval_dir = results_path("step3_evaluation")
    results = run_all_baselines(EPISODE)
    for r in results:
        p = offline_dir / f"baseline_{r.method}_ep{EPISODE}.json"
        offline_dir.mkdir(parents=True, exist_ok=True)
        p.write_text(json.dumps(asdict(r), indent=2))
        print(f"[offline] {r.method:7s} RMSE={r.rmse:.6f}  → {p.name}")
    offline_csv = write_comparison_table(results, offline_dir)
    write_comparison_table(results, eval_dir)
    display(pd.DataFrame([asdict(r) for r in results])[["method", "rmse", "jerk"]])
    print(f"Offline CSV: {offline_csv}")
else:
    print("Skipped offline baselines (RUN_OFFLINE=False).")


In [ ]:
# Cell B — Live UnifoLM dual-process for each bridge
import json
import logging
import multiprocessing as mp
from pathlib import Path

import torch

from src.paths import results_path
from src.step3_dual_thread_mujoco import (
    DualProcessConfig,
    DualProcessController,
    MAX_DURATION_S,
    load_esn_checkpoint_metadata,
    print_run_summary,
    resolve_esn_checkpoint,
    resolve_mjcf_path,
)

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")

live_reports = []
if not RUN_LIVE:
    print("Skipped live dual-process (RUN_LIVE=False).")
else:
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA required for live UnifoLM dual-process.")
    if DURATION_S > MAX_DURATION_S:
        raise ValueError(f"DURATION_S={DURATION_S} exceeds {MAX_DURATION_S}")

    mp.set_start_method("spawn", force=True)
    mjcf = resolve_mjcf_path(None)
    out_dir = results_path("step3_dual_thread")
    out_dir.mkdir(parents=True, exist_ok=True)
    vla_tag = "mock" if MOCK else "live"

    for bridge in BRIDGES:
        ckpt = resolve_esn_checkpoint(None) if bridge == "esn" else Path("unused")
        esn_meta = load_esn_checkpoint_metadata(ckpt) if bridge == "esn" else {}
        config = DualProcessConfig(
            mjcf_path=mjcf,
            esn_checkpoint=str(ckpt),
            mock=MOCK,
            duration_s=DURATION_S,
            control_hz=100.0,
            vla_hz=2.0,
            device=DEVICE,
            record_video=RECORD_VIDEO,
            init_episode=EPISODE,
            bridge=bridge,
        )
        print(f"\n=== LIVE RUN bridge={bridge} mock={MOCK} ===")
        stats = DualProcessController(config).run()
        report = {
            "bridge": bridge,
            "mock_vla": MOCK,
            "steps": stats.steps,
            "mean_step_ms": stats.mean_step_ms,
            "max_step_ms_steady": stats.max_step_ms_steady,
            "p99_step_ms": stats.p99_step_ms,
            "achieved_control_hz": stats.esn_hz,
            "vla_ticks": stats.vla_ticks,
            "gil_bypass_ok": stats.gil_bypass_ok,
            "esn_step2_mse": esn_meta.get("metrics", {}).get("mse") if bridge == "esn" else None,
        }
        report_path = out_dir / f"dual_thread_report_{bridge}_{vla_tag}.json"
        report_path.write_text(json.dumps(report, indent=2))
        live_reports.append(report_path)
        print_run_summary(
            stats,
            control_hz=100.0,
            vla_hz=2.0,
            report_path=report_path,
            profile=False,
            bridge=bridge,
            mock=MOCK,
        )


In [ ]:
# Cell C — Summary table for the plan / paper
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

from src.paths import results_path

eval_dir = results_path("step3_evaluation")
eval_dir.mkdir(parents=True, exist_ok=True)

rows = []
for path in live_reports:
    rows.append(json.loads(Path(path).read_text()))

summary = {
    "offline_baselines_csv": str(offline_csv) if offline_csv else None,
    "live_reports": rows,
}
summary_path = eval_dir / "sim_comparison_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))

if rows:
    display(pd.DataFrame(rows)[
        ["bridge", "mock_vla", "achieved_control_hz", "mean_step_ms", "p99_step_ms", "gil_bypass_ok", "vla_ticks"]
    ])
print(f"Summary: {summary_path}")
print("Sim done → next week: Jetson AGX Thor + G1 via step5_s2r_*.ipynb")
